## 财务 RAG 助手（文本文档版）

### 练习目标
做一个轻量级 **检索增强生成（RAG）** 应用：用自然语言询问本地个人财务文档。

流水线概览：
1. 从 `Documents/finance_docs` 加载可读文本（`.txt` / `.md` / `.csv` 等）
2. 按固定长度切成 **chunks**
3. 用 OpenAI **Embedding** 写入 **Chroma** 向量库
4. 用户在 **Gradio** 里提问 → 检索 Top-k 上下文 → LLM 作答

### 和本课 Week 5 的关系
| 概念 | 本作业位置 |
|------|------------|
| 文档加载与切块 | `load_documents` / `chunk_documents` |
| Embedding + Vector DB | `create_embeddings` / `store_vectors` / Chroma |
| RAG 生成 | `rag_pipeline` + `SYSTEM_PROMPT` |
| 聊天 UI | `gr.ChatInterface` |

### 怎么跑
1. `.env` 配置 `OPENAI_API_KEY`
2. 把财务文本放进家目录下的 `Documents/finance_docs`
3. 从上到下运行单元格；最后会启动 Gradio 聊天窗

> 本版本刻意避开 PDF 解析，只处理可读文本文档。


In [ ]:
# ========== 安装运行依赖 ==========

# !pip：在 Notebook 里临时安装 openai / chromadb / gradio / python-dotenv
!pip install openai chromadb gradio python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 3.0 MB/s  0:00:023.1 MB/s eta 0:00:01:01


In [40]:
# ========== 导入：标准库 + Chroma + Gradio + OpenAI ==========

# os：读环境变量（API Key）
import os
# uuid：给每条向量生成唯一 id
import uuid
# chromadb：向量数据库客户端
import chromadb
# gradio：聊天界面
import gradio as gr
# Path：拼出家目录下的文档路径
from pathlib import Path
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI：Embedding + Chat Completions
from openai import OpenAI


In [ ]:
# ========== 读取 OPENAI_API_KEY 并创建客户端 ==========

# 把 .env 写入进程环境
load_dotenv()

# 取出密钥字符串
api_key = os.getenv("OPENAI_API_KEY")

# 有密钥 → 构造客户端；没有 → 立刻报错，避免后面静默失败
if api_key:
    client = OpenAI(api_key=api_key)
else:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")


In [ ]:
# ========== 路径与集合名常量 ==========

# 财务文档目录：~/Documents/finance_docs
DOCUMENT_FOLDER = Path.home() / "Documents" / "finance_docs"
# Chroma collection 名称（逻辑库名）
CHROMA_COLLECTION = "finance_knowledge_base"


In [ ]:
# ========== 确保文档目录存在，并提示用户往哪放文件 ==========

def ensure_documents_folder():
    # parents=True：中间目录一并创建；exist_ok=True：已存在不报错
    DOCUMENT_FOLDER.mkdir(parents=True, exist_ok=True)
    # 提示文案保持英文原样
    print("Place finance documents in:")
    print(DOCUMENT_FOLDER)


In [ ]:
# ========== 初始化内存版 Chroma 并拿到（或创建）集合 ==========

def initialize_vector_db():
    # Client()：默认进程内/临时客户端（非 PersistentClient）
    chroma_client = chromadb.Client()
    # 按名称获取或新建 collection
    collection = chroma_client.get_or_create_collection(
        name=CHROMA_COLLECTION
    )
    return collection


In [ ]:
# ========== 遍历文档夹：读文本，跳过失败文件 ==========

def load_documents():
    # 累积 {"text","source"} 字典
    documents = []

    # 只扫一层目录（非递归）
    for file in DOCUMENT_FOLDER.iterdir():
        try:
            # errors="ignore"：遇到坏字节不中断
            with open(file, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            # 空文件跳过
            if text.strip():
                documents.append({
                    "text": text,
                    "source": file.name
                })
        except Exception:
            # 读失败只打印文件名，继续下一个
            print("Skipping file:", file.name)

    print("Documents loaded:", len(documents))
    return documents


In [ ]:
# ========== 固定窗口切块：每 chunk_size 字符一段 ==========

def chunk_documents(documents, chunk_size=500):
    chunks = []

    for doc in documents:
        # 取出全文
        text = doc["text"]

        # 步长 = chunk_size，无重叠的简单切片
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i + chunk_size]

            # 保留来源文件名，便于溯源（当前入库未写 metadata，结构先留下）
            chunks.append({
                "text": chunk,
                "source": doc["source"]
            })

    print("Chunks created:", len(chunks))
    return chunks


In [ ]:
# ========== 批量调用 Embedding API ==========

def create_embeddings(texts):
    # text-embedding-3-small：便宜够用的句向量模型
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    # 抽出每条 embedding 向量
    embeddings = [item.embedding for item in response.data]
    return embeddings


In [ ]:
# ========== 把文本块与向量写入 Chroma ==========

def store_vectors(chunks, embeddings, collection):
    ids = []
    documents = []

    # 为每块生成 UUID，并收集纯文本
    for chunk in chunks:
        ids.append(str(uuid.uuid4()))
        documents.append(chunk["text"])

    # documents / embeddings / ids 三个列表必须等长对齐
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )


In [42]:
# ========== 先初始化全局 collection，供后续建库与检索共用 ==========

collection = initialize_vector_db()


In [ ]:
# ========== 一键建库：加载 → 切块 → 嵌入 → 入库 ==========

def build_vector_database():
    # 读本地财务文档
    documents = load_documents()

    # 没有文档就提前返回，避免空 embedding 调用
    if not documents:
        print("No documents found")
        return

    # 切成固定长度块
    chunks = chunk_documents(documents)
    # 只要文本列表给 Embedding API
    texts = [chunk["text"] for chunk in chunks]

    # 算向量
    embeddings = create_embeddings(texts)
    # 写入全局 collection
    store_vectors(chunks, embeddings, collection)

    print("Vector database built successfully")


In [ ]:
# ========== 把用户问题也变成同一 embedding 空间的向量 ==========

def embed_query(query):
    # 单条 query 也走同一个 embedding 模型，保证可比
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    # 取第一条向量
    return response.data[0].embedding


In [ ]:
# ========== 向量近邻检索：拼出上下文字符串 ==========

def retrieve_context(query_vector, collection, k=3):
    # query_embeddings 要包一层 list；n_results=Top-k
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k
    )

    # results["documents"][0] 是本次查询命中的文本列表
    context = "\n\n".join(results["documents"][0])
    return context


In [ ]:
# ========== 系统提示：财务助手角色 + 只能依据文档作答 ==========

# 英文 prompt 保持原样，勿翻译（影响模型行为）
SYSTEM_PROMPT = """
You are a financial analysis assistant.

Use the provided finance documents to answer the question.

If the answer is not in the documents,
say you do not know.
"""


In [ ]:
# ========== 调用聊天模型：system=提示+上下文，user=问题 ==========

def generate_answer(question, context):
    # messages：标准 Chat Completions 格式
    messages = [
        {
            "role": "system",
            # 把 SYSTEM_PROMPT 与检索到的 Context 拼在一起
            "content": SYSTEM_PROMPT + "\n\nContext:\n" + context
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # gpt-4o-mini：便宜、适合短问答
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # 返回助手文本
    return response.choices[0].message.content


In [ ]:
# ========== 再次初始化 collection（与第 11 格相同调用，保持原流程）==========

collection = initialize_vector_db()


In [ ]:
# ========== RAG 管道：问题 → 向量 → 检索 → 生成 ==========

def rag_pipeline(question):
    # 问题向量化
    query_vector = embed_query(question)
    # 取 Top-k 上下文
    context = retrieve_context(query_vector, collection)
    # 带上下文生成最终答案
    answer = generate_answer(question, context)
    return answer


In [ ]:
# ========== Gradio 聊天回调：忽略 history，每轮独立走 RAG ==========

def chat(message, history):
    # message 是当前用户输入；history 未使用（保持原签名以兼容 ChatInterface）
    answer = rag_pipeline(message)
    return answer


In [ ]:
# ========== 启动前准备：建文件夹 + 灌入向量库 ==========

# 确保目录存在并打印路径提示
ensure_documents_folder()
# 读取文档、切块、嵌入、写入 Chroma
build_vector_database()


Place finance documents in:
/home/steve/Documents/finance_docs
Documents loaded: 5
Chunks created: 5
Vector database built successfully


In [41]:
# ========== 启动 Gradio ChatInterface ==========

# type="messages"：使用 OpenAI 风格消息列表；launch 打开浏览器
view = gr.ChatInterface(
    chat,
    type="messages"
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
